# Install dependencies

In [ ]:
!pip install pygeohash category_encoders lightgbm xgboost optuna scikit-learn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
import warnings
import re
import optuna
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from category_encoders import TargetEncoder

import matplotlib.pyplot as plt

# Load data : google drive
change this cell for own data loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/gridlock-2.0/datasets"
train = pd.read_csv(f"{path}/train.csv")
test  = pd.read_csv(f"{path}/test.csv")

print(f"Train: {train.shape}   Test: {test.shape}")
train.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (77299, 11)   Test: (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


# Data analysis

In [ ]:
print("--- dtypes ---")
print(train.dtypes)
print("\n--- missing % ---")
print((train.isnull().mean() * 100).round(2))
print("\n--- target stats ---")
print(train['demand'].describe())

--- dtypes ---
Index              int64
geohash           object
day                int64
timestamp         object
demand           float64
RoadType          object
NumberofLanes      int64
LargeVehicles     object
Landmarks         object
Temperature      float64
Weather           object
dtype: object

--- missing % ---
Index            0.00
geohash          0.00
day              0.00
timestamp        0.00
demand           0.00
RoadType         0.78
NumberofLanes    0.00
LargeVehicles    0.00
Landmarks        0.00
Temperature      3.23
Weather          1.03
dtype: float64

--- target stats ---
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64


In [ ]:
print(train['Weather'].value_counts())
print(train['RoadType'].value_counts())
print(train['NumberofLanes'].value_counts())
print(train['LargeVehicles'].value_counts())
print(train['Landmarks'].value_counts())

Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64
RoadType
Residential    69230
Street          3909
Highway         3560
Name: count, dtype: int64
NumberofLanes
1    27411
2    24127
3    23919
4      926
5      916
Name: count, dtype: int64
LargeVehicles
Not Allowed    50673
Allowed        26626
Name: count, dtype: int64
Landmarks
Yes    52042
No     25257
Name: count, dtype: int64


In [ ]:
avg_temp_by_day = train.groupby('day')['Temperature'].mean()
print("Average Temperature by Day:")
print(avg_temp_by_day)

Average Temperature by Day:
day
48    16.422095
49    16.257708
Name: Temperature, dtype: float64


# impute road type: most frequent

In [ ]:
train['RoadType'].fillna(train['RoadType'].mode()[0], inplace=True)
test['RoadType'].fillna(test['RoadType'].mode()[0], inplace=True)

# Feature Engineering and imputatinos

In [ ]:
# -------------------------Timestamp-----------------------------
class TimestampEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self

    def transform(self, X):
        # X = X.copy()
        ts = pd.to_datetime(X['timestamp'], format='%H:%M', errors='coerce')
        X['hour']   = ts.dt.hour
        X['minute'] = ts.dt.minute

        # minutes passed overall
        X['overall_minutes']=X['day']*24*60+X['hour']*60+X['minute']

        # minutes passed day wise
        X['day_minutes']=X['hour']*60+X['minute']

        # cyclic minutes
        X['day_minutes_sin'] = np.sin(2 * np.pi * X['day_minutes'] / (24 * 60))
        X['day_minutes_cos'] = np.cos(2 * np.pi * X['day_minutes'] / (24 * 60))
        # cyclic hour
        X['day_hour_sin'] = np.sin(2 * np.pi * X['hour'] / 24)
        X['day_hour_cos'] = np.cos(2 * np.pi * X['hour'] / 24)

        # time slot in a day (0-95)
        X['time_slot_15'] = X['hour'] * 4 + (X['minute'] // 15)
        X['time_slot_30'] = X['hour'] * 2 + (X['minute'] // 30)

        # Peak-hour flags
        X['is_early_morning_peak'] = ((X['hour'] > 5) & (X['hour'] < 7)).astype(int)
        X['is_morning_peak'] = ((X['hour'] > 7)  & (X['hour'] <= 10)).astype(int)
        X['is_afternoon_peak'] = ((X['hour'] > 10) & (X['hour'] <= 16)).astype(int)
        X['is_evening_peak'] = ((X['hour'] > 16) & (X['hour'] <= 18)).astype(int)
        X['is_post_evening_peak'] = ((X['hour'] > 18) & (X['hour'] <= 22)).astype(int)
        X['is_night']        = ((X['hour'] > 22) | (X['hour'] <= 5)).astype(int)
        # Business hours flag
        X['is_business_hours'] = ((X['hour'] >= 9) & (X['hour'] <= 17)).astype(int)
        # Lunch hour flag
        X['is_lunch'] = ((X['hour'] >= 12) & (X['hour'] <= 14)).astype(int)

        # Interaction: day × time_slot
        X['day_time_slot_15'] = X['day'].astype(str) + '_' + X['time_slot_15'].astype(str)
        X['day_time_slot_30'] = X['day'].astype(str) + '_' + X['time_slot_30'].astype(str)

        return X.drop(columns=['timestamp', 'minute'])


# --------------------------geohash engineering---------------------------------
class GeohashEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Cache lat/lon decode to avoid repeat calls
        unique_gh = X['geohash'].dropna().unique()
        self._latlon = {gh: pgh.decode(gh) for gh in unique_gh}
        return self

    def transform(self, X):
        # X = X.copy()

        X['lat'] = X['geohash'].map(lambda x: self._latlon.get(x, (np.nan, np.nan))[0])
        X['lon'] = X['geohash'].map(lambda x: self._latlon.get(x, (np.nan, np.nan))[1])

        # Distance from city center
        CENTER_LAT = X['lat'].median()
        CENTER_LON = X['lon'].median()
        X['dist_from_center'] = np.sqrt(
            (X['lat'] - CENTER_LAT)**2 + (X['lon'] - CENTER_LON)**2
        )

        # Hierarchical geohash prefixes
        X['geo_l5'] = X['geohash'].str[:5]
        X['geo_l4'] = X['geohash'].str[:4]
        X['geo_l3'] = X['geohash'].str[:3]

        # Interaction keys (geo5 × time features)
        X['geo5_time_slot_15'] = X['geo_l5'] + '_' + X['time_slot_15'].astype(str)
        X['geo5_time_slot_30'] = X['geo_l5'] + '_' + X['time_slot_30'].astype(str)
        X['geo5_hour']      = X['geo_l5'] + '_' + X['hour'].astype(str)

        # geo5 X peak flags
        X['geo5_early_morning_peak'] = X['geo_l5'] + '_' + X['is_early_morning_peak'].astype(str)
        X['geo5_morning_peak'] = X['geo_l5'] + '_' + X['is_morning_peak'].astype(str)
        X['geo5_afternoon_peak'] = X['geo_l5'] + '_' + X['is_afternoon_peak'].astype(str)
        X['geo5_evening_peak'] = X['geo_l5'] + '_' + X['is_evening_peak'].astype(str)
        X['geo5_post_evening_peak'] = X['geo_l5'] + '_' + X['is_post_evening_peak'].astype(str)
        X['geo5_night_peak']        = X['geo_l5'] + '_' + X['is_night'].astype(str)

        # Drop raw geohash & helper cols not needed downstream
        return X


# -----------------------------temp, weather imputation- ---------------------------
class HierarchicalImputer(BaseEstimator, TransformerMixin):
    TEMP_KEYS    = ['geo5_time_slot_15','geo5_time_slot_30', 'geo5_hour']
    WEATHER_KEYS = ['geo5_time_slot_15','geo5_time_slot_30', 'geo5_hour']
    WEATHER_FALLBACK = 'Clear'

    def __init__(self):
        self.temp_stats_    = {}
        self.weather_stats_ = {}
        self.temp_global_   = None
        self.weather_global_= None

    def fit(self, X, y=None):
        # ── Temperature ───────────────────────
        temp   = X['Temperature'].copy()
        filled = temp.copy()
        for kc in self.TEMP_KEYS:
            stat_map = filled.groupby(X[kc]).median().to_dict()
            self.temp_stats_[kc] = stat_map
            filled = filled.fillna(X[kc].map(stat_map))
        self.temp_global_ = temp.mean()

        # ── Weather ─────────────────────────
        weather = X['Weather'].copy()
        filled  = weather.copy()
        for kc in self.WEATHER_KEYS:
            stat_map = (
                filled.groupby(X[kc])
                      .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
                      .to_dict()
            )
            self.weather_stats_[kc] = stat_map
            filled = filled.fillna(X[kc].map(stat_map))
        modes = weather.dropna().mode()
        self.weather_global_ = modes.iloc[0] if not modes.empty else self.WEATHER_FALLBACK

        return self

    def transform(self, X):
        X = X.copy()

        # ── Impute Temperature ─────────────────────────────────────────────
        temp = X['Temperature'].copy()
        for kc in self.TEMP_KEYS:
            temp = temp.fillna(X[kc].map(self.temp_stats_[kc]))
        X['Temperature'] = temp.fillna(self.temp_global_)

        # ── Impute Weather ─────────────────────────────────────────────────
        weather = X['Weather'].copy()
        for kc in self.WEATHER_KEYS:
            weather = weather.fillna(X[kc].map(self.weather_stats_[kc]))
        X['Weather'] = weather.fillna(self.weather_global_)

        return X


# ---------------------------temp,weather,roadtype,noOfLanes engineering -----------------------------
class TempEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self

    def transform(self, X):
      # temperature buckets
      X['temp_bucket'] = pd.cut(
          X['Temperature'],
          bins=[-np.inf, 5, 15, 25, 35, np.inf],
          labels=['freezing', 'cold', 'comfortable', 'warm', 'hot']
      )

      # temperature X weather
      X['weather_temp_bucket'] = X['Weather'].astype(str) + '_' + X['temp_bucket'].astype(str)

      # temperature X geohash
      X['geo_temp_bucket'] = X['geo_l5'].astype(str) + '_' + X['temp_bucket'].astype(str)

      # weather X geo5
      X['weather_geo5'] = X['Weather'].astype(str) + '_' + X['geo_l5'].astype(str)

      # weather
      X['is_cold_weather']=(X['Weather']=='Snowy').astype(int)

      # road type
      X['is_residential_road']=(X['RoadType']=='Residential').astype(int)

      # number of lanes
      X['is_more_lanes']=(X['NumberofLanes']>3).astype(int)

      return X

# Colum transformer

In [ ]:
SCALE_COLS = [
    'day', 'overall_minutes', 'day_minutes',
    'time_slot_15', 'time_slot_30',
    'lat', 'lon', 'dist_from_center',
    'Temperature',
]

# Already in [0,1] or small integers — no scaling needed
PASSTHROUGH_COLS = [
    'day_minutes_sin', 'day_minutes_cos',
    'day_hour_sin', 'day_hour_cos',
    'hour',
    'is_early_morning_peak', 'is_morning_peak', 'is_afternoon_peak',
    'is_evening_peak', 'is_post_evening_peak', 'is_night',
    'is_business_hours', 'is_lunch',
    'is_cold_weather', 'is_residential_road', 'is_more_lanes',
    'NumberofLanes'
]

# Low-cardinality → OHE
OHE_COLS = ['RoadType', 'Weather','temp_bucket']

# Ordinal encoding-> binary
ORD_ENC_COLS = ['LargeVehicles', 'Landmarks']

# High-cardinality string keys → TargetEncoder
TARGET_ENC_COLS = [
    'geo_l3', 'geo_l4', 'geo_l5', 'geohash',
    'geo5_time_slot_15', 'geo5_time_slot_30', 'geo5_hour',
    'geo5_early_morning_peak', 'geo5_morning_peak', 'geo5_afternoon_peak',
    'geo5_evening_peak', 'geo5_post_evening_peak', 'geo5_night_peak',
    'day_time_slot_15', 'day_time_slot_30',
    'geo_temp_bucket', 'weather_geo5', 'weather_temp_bucket',
]

# ── ColumnTransformer ──────────────────────────────────────────────────────────

col_transformer = ColumnTransformer(
    transformers=[
        ('scaler',
         StandardScaler(),
         SCALE_COLS),

        ('ohe',
         OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
         OHE_COLS),

        ('ord_enc',
         OrdinalEncoder(categories=[['Not Allowed', 'Allowed'],['No', 'Yes']],handle_unknown='use_encoded_value',unknown_value=-1),
         ORD_ENC_COLS),

        ('target_enc',
         TargetEncoder(smoothing=10, min_samples_leaf=5),
         TARGET_ENC_COLS),

        ('passthrough',
         'passthrough',
         PASSTHROUGH_COLS),
    ],
    remainder='drop',               # anything not listed above is dropped
    verbose_feature_names_out=False,
    n_jobs=-1,
)

# Preprocess pipeline

In [ ]:
preproc_pipeline = Pipeline(steps=[
    ('ts_eng',  TimestampEngineer()),
    ('geo_eng', GeohashEngineer()),
    ('impute',  HierarchicalImputer()),
    ('other_eng', TempEngineer()),
    ('col_tr',  col_transformer),
])

print('Preprocessing pipeline defined.')

Preprocessing pipeline defined.


# Feature selection: top log2(n)

In [ ]:
import math
from sklearn.base import BaseEstimator, TransformerMixin

class Log2FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        var_threshold: float = 0.01,     # columns with variance < this are dropped
        corr_threshold: float = 0.95,    # Pearson |r| above this = redundant
        probe_estimator=None,            # quick LGBM for importance ranking
        random_state: int = 42,
    ):
        self.var_threshold    = var_threshold
        self.corr_threshold   = corr_threshold
        self.probe_estimator  = probe_estimator
        self.random_state     = random_state

        # Filled during fit()
        self.selected_indices_  = None   # final column positions to keep
        self.n_original_        = None
        self.n_selected_        = None

    def fit(self, X, y=None):
        X = np.array(X, dtype=float)
        n_samples, n_features = X.shape
        self.n_original_ = n_features
        target_k = max(1, (int(math.log2(n_features))*5))
        # target_k = max(1, 20)

        # ── Stage 1: Variance filter ──────────────────────────────────────
        variances = np.var(X, axis=0)
        after_var = np.where(variances >= self.var_threshold)[0]
        print(f"[FeatureSel] Stage 1 – variance filter: "
              f"{n_features} → {len(after_var)} features "
              f"(dropped {n_features - len(after_var)} near-zero-var)")

        # ── Stage 2: Correlation filter(pearson correlation) ───────────────────────────────────
        X_var = X[:, after_var]
        corr  = np.corrcoef(X_var, rowvar=False)
        np.fill_diagonal(corr, 0.0)          # ignore self-correlation

        drop_local = set()
        for i in range(corr.shape[0]):
            if i in drop_local:
                continue
            for j in range(i + 1, corr.shape[1]):
                if j in drop_local:
                    continue
                if abs(corr[i, j]) >= self.corr_threshold:
                    # Drop the one with lower variance (keep more informative)
                    if variances[after_var[i]] < variances[after_var[j]]:
                        drop_local.add(i)
                    else:
                        drop_local.add(j)

        keep_local = [i for i in range(len(after_var)) if i not in drop_local]
        after_corr = after_var[keep_local]
        print(f"[FeatureSel] Stage 2 – correlation filter: "
              f"{len(after_var)} → {len(after_corr)} features "
              f"(dropped {len(after_var) - len(after_corr)} correlated)")

        # ── Stage 3: LGBM importance (top log2) ───────────────────────────
        X_corr = X[:, after_corr]

        if self.probe_estimator is None:
            probe = lgb.LGBMRegressor(
                n_estimators=200,
                learning_rate=0.1,
                num_leaves=63,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=self.random_state,
                verbose=-1,
                n_jobs=-1,
            )
        else:
            probe = self.probe_estimator

        probe.fit(X_corr, y)
        importances = probe.feature_importances_          # shape: (len(after_corr),)

        # Rank by importance, pick top log2(n_original)
        sorted_local = np.argsort(importances)[::-1]     # descending
        selected_local = sorted_local[:target_k]
        selected_local.sort()                             # restore column order

        self.selected_indices_ = after_corr[selected_local]
        self.n_selected_       = len(self.selected_indices_)

        print(f"[FeatureSel] Stage 3 – LGBM importance: "
              f"{len(after_corr)} → {self.n_selected_} features "
              f"(target = log2({n_features})*5 ≈ {target_k})")
        print(f"[FeatureSel] Final selection: {self.n_selected_} / {n_features} features kept.")
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        return X[:, self.selected_indices_]

    def get_support_mask(self):
        mask = np.zeros(self.n_original_, dtype=bool)
        mask[self.selected_indices_] = True
        return mask

# Train test split , target transform

In [ ]:
# Drop unwanted colums
DROP_RAW = ['Index']

X = train.drop(columns=DROP_RAW + ['demand'])
y = train['demand'].copy()

X_test_raw = test.drop(columns=[c for c in DROP_RAW if c in test.columns])

# Log1p transform on target — demand is likely right-skewed
y_log = np.log1p(y)

print(f"X: {X.shape}   y skew before: {y.skew():.2f}  after log1p: {y_log.skew():.2f}")
print("Columns:", X.columns.tolist())

X: (77299, 9)   y skew before: 3.73  after log1p: 2.97
Columns: ['geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']


# hyperparameter tuning using optuna
will take some time

In [ ]:
N_TRIALS  = 100
N_FOLDS_TUNE  = 3
RANDOM_SEED   = 42

def objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 500, 3000, step=100),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 31, 255),
        'max_depth':          trial.suggest_int('max_depth', 4, 12),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_split_gain':     trial.suggest_float('min_split_gain', 0.0, 0.5),
        'random_state':       RANDOM_SEED,
        'verbose':            -1,
        'n_jobs':             -1,
    }

    kf    = KFold(n_splits=N_FOLDS_TUNE, shuffle=True, random_state=RANDOM_SEED)
    r2_scores = []

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y_log.iloc[tr_idx], y_log.iloc[va_idx]

        # Fit preprocessing on train fold only (avoids leakage)
        pp = preproc_pipeline

        X_tr_t = pp.fit_transform(X_tr, y_tr)
        X_va_t = pp.transform(X_va)

        # feature selection
        selector = Log2FeatureSelector(random_state=RANDOM_SEED)
        selector.fit(X_tr_t, y_tr)
        X_tr_t = selector.transform(X_tr_t)
        X_va_t = selector.transform(X_va_t)

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr_t, y_tr,
            eval_set=[(X_va_t, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )

        preds = model.predict(X_va_t)
        r2  = r2_score(y_va, preds)
        r2_scores.append(r2)

    return np.mean(r2_scores)


study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest R2 score (log-scale): {study.best_value:.5f}")
print("Best params:", study.best_params)


  0%|          | 0/80 [00:00<?, ?it/s]

[FeatureSel] Stage 1 – variance filter: 55 → 36 features (dropped 19 near-zero-var)
[FeatureSel] Stage 2 – correlation filter: 36 → 29 features (dropped 7 correlated)
[FeatureSel] Stage 3 – LGBM importance: 29 → 20 features (target = log2(55) ≈ 20)
[FeatureSel] Final selection: 20 / 55 features kept.
[FeatureSel] Stage 1 – variance filter: 55 → 36 features (dropped 19 near-zero-var)
[FeatureSel] Stage 2 – correlation filter: 36 → 29 features (dropped 7 correlated)
[FeatureSel] Stage 3 – LGBM importance: 29 → 20 features (target = log2(55) ≈ 20)
[FeatureSel] Final selection: 20 / 55 features kept.
[FeatureSel] Stage 1 – variance filter: 55 → 36 features (dropped 19 near-zero-var)
[FeatureSel] Stage 2 – correlation filter: 36 → 29 features (dropped 7 correlated)
[FeatureSel] Stage 3 – LGBM importance: 29 → 20 features (target = log2(55) ≈ 20)
[FeatureSel] Final selection: 20 / 55 features kept.
[FeatureSel] Stage 1 – variance filter: 55 → 36 features (dropped 19 near-zero-var)
[FeatureSe

# final model training and predicting

In [ ]:
best_params   = study.best_params
best_params.update({'random_state': RANDOM_SEED, 'verbose': -1, 'n_jobs': -1})

# Prepare data for training a single model
print("\nFitting preprocessing pipeline on full training data...")
pp = preproc_pipeline
X_train_transformed = pp.fit_transform(X, y_log)
X_test_transformed  = pp.transform(X_test_raw)

# feature selection
selector = Log2FeatureSelector(random_state=RANDOM_SEED)
selector.fit(X_train_transformed, y_log)
X_train_transformed = selector.transform(X_train_transformed)
X_test_transformed  = selector.transform(X_test_transformed)
print(f"Training on {X_train_transformed.shape[1]} features after log2 selection.")

# Train a single model with best parameters on the full training data
print("Training final LightGBM model...")
model = lgb.LGBMRegressor(**best_params)
model.fit(X_train_transformed, y_log)

# Make predictions on the full training data (for OOF R2, if desired) and test data
print("Making predictions...")
oof_preds   = model.predict(X_train_transformed)
test_preds  = model.predict(X_test_transformed)

# Overall OOF score (calculated on the full training data predictions)
oof_r2 = r2_score(y_log, oof_preds)
print(f"\n★ Full Training R2 Score (log-scale): {oof_r2:.5f}")

# Inverse transform predictions to original scale
oof_demand  = np.expm1(oof_preds)
test_demand = np.expm1(test_preds)

# R2 score on original scale
orig_r2 = r2_score(y, oof_demand)
print(f"★ Full Training R2 Score (original scale):    {orig_r2:.5f}")



Fitting preprocessing pipeline on full training data...
[FeatureSel] Stage 1 – variance filter: 55 → 36 features (dropped 19 near-zero-var)
[FeatureSel] Stage 2 – correlation filter: 36 → 29 features (dropped 7 correlated)
[FeatureSel] Stage 3 – LGBM importance: 29 → 25 features (target = log2(55) ≈ 25)
[FeatureSel] Final selection: 25 / 55 features kept.
Training on 25 features after log2 selection.
Training final LightGBM model...
Making predictions...

★ Full Training R2 Score (log-scale): 0.95471
★ Full Training R2 Score (original scale):    0.96059


# Generate submission file
change target file path to save at that location

In [ ]:
test_demand_clipped = np.clip(test_demand, 0, None)

submission = pd.DataFrame({
    'Index':  test['Index'],
    'demand': test_demand_clipped
})

submission.to_csv(f"{path}/output.csv", index=False)
print("Saved submission!")
submission.head()

Saved submission!


,Index,demand
0,0,0.038019
1,1,0.038038
2,2,0.013259
3,3,0.019522
4,4,0.040360
